# Phase 4a: seven-dose response with 2x and 4x (exploratory)

Reads `results/phase4a_2x4x_models.csv` and `results/phase4a_2x4x_score.csv` (written by `python -m zoo.phase4a_2x4x`) and the naive
models' Gate 3 assay summaries. Trains nothing. Writes `reports/figures/phase4a_2x4x.png`.

In [ ]:
from pathlib import Path
import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MOD = pd.read_csv(ROOT / "results/phase4a_2x4x_models.csv")
SC = pd.read_csv(ROOT / "results/phase4a_2x4x_score.csv")
DOSES = [1, 2, 4, 8, 16, 32, 64]
tn = [json.loads((ROOT / f"runs/assay-plant_lr3e-4-naive-s{k}-a{a}/summary.json").read_text())["t_cross"] for k in range(5) for a in range(3)]
print("naive crossings:", min(tn), "-", max(tn))
SC

In [ ]:
INK, MUTED, SEED, MED, NEW = "#222222", "#6b6b6b", "#9aa4b1", "#1f4e79", "#c0504d"
fig, (A, B) = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)
jit = {0: -0.12, 1: -0.06, 2: 0.0, 3: 0.06, 4: 0.12}
x = range(len(DOSES))
for ax, col, ylab in ((A, "t_median", "crossing step (median of 3 assay seeds)"),
                      (B, "fold", "times faster than same-seed naive (2^-L)")):
    if col == "fold":
        MOD["fold"] = 2 ** (-MOD.L)
    for k, g in MOD.groupby("train_seed"):
        g = g.set_index("dose").loc[DOSES]
        ax.plot([i + jit[k] for i in x], g[col], color=SEED, lw=1, marker="o", ms=4, zorder=2,
                label="one training seed" if k == 0 else None)
    med = MOD.groupby("dose")[col].median().loc[DOSES]
    ax.plot(list(x), med, color=MED, lw=2, marker="s", ms=6, zorder=3, label="median of 5 seeds")
    for i, d in enumerate(DOSES):
        if d in (2, 4):
            ax.axvspan(i - 0.3, i + 0.3, color=NEW, alpha=0.08, lw=0, zorder=1)
    ax.set_xticks(list(x), [f"{d}x" for d in DOSES]); ax.set_xlabel("counter-training dose (shaded: new, 2x and 4x)")
    ax.set_ylabel(ylab); ax.spines[["top", "right"]].set_visible(False); ax.grid(axis="y", color="#e5e5e5", lw=0.6)
A.set_ylim(0, 120); B.set_ylim(0, 35)
A.text(0.98, 0.80, f"naive models cross at {min(tn)}-{max(tn)}", transform=A.transAxes, ha="right", va="top", fontsize=8, color=MUTED)
A.set_title("A  Frozen assay crossing, 5 training seeds", loc="left", fontsize=10)
B.set_title("B  Speed-up against naive (lower = slower relearning)", loc="left", fontsize=10)
A.legend(frameon=False, fontsize=8, loc="upper right")
fig.savefig(ROOT / "reports/figures/phase4a_2x4x.png", dpi=150)
MOD.pivot(index="train_seed", columns="dose", values="t_median")